In [ ]:
import os
import subprocess

configs_dir = "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/"

# NNMFit v0.5.0 venv: includes CombinedTemplateFlux (muontemplate_combined_hese)
venv_bin = "/data/user/tvaneede/software/py_venvs/py3-v4.2.1_nnmfit-v0.5.0/bin"
python = f"{venv_bin}/python"
script = f"{venv_bin}/calculate_graph.py"
make_histogram_script = f"{venv_bin}/make_histogram.py"

In [ ]:
main_out_dir = f"/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks/unblind/step1_hese_flux_reunblind/graphs"
os.makedirs(main_out_dir, exist_ok=True)

In [ ]:
model = "mcd-simpletopology_flux-hese_feat-11features_plus_rloglmilli_econf_evtgen"
threshold = "bdt1_0.333333_bdt2_0.366667_length_10"

# fit directory, see dag_scans/flavor_globalfit/hese/unblind/step1_hese_flux_try2/data_fits.sh
fit_dir = "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step1_hese_flux_try2/data_fits"

In [ ]:
# same settings as the fit (data_fits.sh), the minimizer override does not affect histograms
DEFAULT_OVERRIDE_CONFIGS = [
    "override/livetime/hese_livetime_13yr_asr.cfg",
]
# astro parametrization of the fit (astro_norm, gamma_astro);
# the muon template (muontemplate_combined_hese) comes from main_muonconvpromptcombined.cfg
DEFAULT_OVERRIDE_COMPONENTS = [
    "override/components/astro_SPL_no_inel_no_flavor.yaml",
]
# equivalent 3-flavor parametrization (total_astro_norm, a, b, gamma), only for the flavor plots
OVERRIDE_COMPONENTS_3FLAVOR = [
    "override/components/astro_SPL_3flavor_no_inel.yaml",
]

DEFAULT_NO_SYSTEMATICS = ["override/systematics/NoSystematics_hese.cfg"]
DEFAULT_SYSTEMATICS = [f"override/systematics/hese/Snowstorm_Gradients_hese_HESEBestfit_SPL_reunblind/{model}/{threshold}_wpriors.cfg"]
# old passing fractions: previous datasets (data_v3) and Snowstorm gradients, for comparison
OLD_PF_DATASETS = [f"override/datasets_hese/data_v3/{model}/{threshold}.cfg"]
OLD_PF_SYSTEMATICS = [f"override/systematics/hese/Snowstorm_Gradients_hese_HESEBestfit_SPL/{model}/{threshold}_wpriors.cfg"]

CONFIGS = []

In [ ]:
# best fit with the PassingHESE passing fractions, see step1_hese_flux_try2/data_fits.sh
scan = "nbestfit50_all_param_2sigma_1D_10steps_PassingHESE"

# settings shared by both graphs; all analysis configs (written by run_post_fit_scripts.ipynb) are in one
# folder and contain both astro_norm and total_astro_norm/a/b/gamma, each graph only uses its own
_COMMON = {
    "analysis_config": f"analysis_configs/data/hese/step1_hese_flux_reunblind/plotting/{scan}/{scan}.yaml",
    "main_config" : "main_muonconvpromptcombined.cfg",
    "override_configs": [f"override/datasets_hese/data_v5/{model}/{threshold}_PassingHESE.cfg",
                         f"override/binning/hese/10bdtprod_threshold_0.122.cfg"] + DEFAULT_OVERRIDE_CONFIGS,
    "override_parameters": None,
    "systematics": DEFAULT_SYSTEMATICS,
    # fit directory to compare the data histogram with (None: no comparison)
    "fit_dir": os.path.join(fit_dir, scan),
}

# "components" maps histogram name -> analysis config suffix ({scan}_{suffix}.yaml)
_MC_COMPONENTS = {c: c for c in ["Conv", "Prompt", "Muon", "MuonConv", "MuonPrompt"]}

CONFIGS += [
    # parametrization used in the fit
    {
        **_COMMON,
        "name": scan,
        "override_components": DEFAULT_OVERRIDE_COMPONENTS,
        "components": {"Astro": "Astro", **_MC_COMPONENTS},
    },
    # 3-flavor parametrization at the equivalent 1:1:1 point, for the flavor plots
    {
        **_COMMON,
        "name": f"{scan}_3flavor",
        "override_components": OVERRIDE_COMPONENTS_3FLAVOR,
        "components": {
            "Astro": "Astro_3flavor",
            "Astro_NuE": "Astro_NuE",
            "Astro_NuMu": "Astro_NuMu",
            "Astro_NuTau": "Astro_NuTau",
            **_MC_COMPONENTS,
        },
    },
    # fit parametrization with the old passing fractions: previous datasets (data_v3) and Snowstorm gradients
    {
        **_COMMON,
        "name": f"{scan}_old_passing_fractions",
        "override_configs": OLD_PF_DATASETS + [f"override/binning/hese/10bdtprod_threshold_0.122.cfg"] + DEFAULT_OVERRIDE_CONFIGS,
        "override_components": DEFAULT_OVERRIDE_COMPONENTS,
        "systematics": OLD_PF_SYSTEMATICS,
        "fit_dir": None,  # different data sample (data_HESE_pass2_v3) than the fit
        "components": {"Astro": "Astro", **_MC_COMPONENTS},
    },
]

# fit parametrization with the crystal density scaling (ice_crystal) fixed by hand, all other inputs as the best fit
CRYSTAL_VALUES = [0.9, 1.0, 1.1, 1.2]
CONFIGS += [
    {
        **_COMMON,
        "name": f"{scan}_ice_crystal_{crystal}",
        "analysis_config": f"analysis_configs/data/hese/step1_hese_flux_reunblind/plotting/{scan}/{scan}_ice_crystal_{crystal}.yaml",
        "override_components": DEFAULT_OVERRIDE_COMPONENTS,
        "components": {"Astro": "Astro", **_MC_COMPONENTS},
    }
    for crystal in CRYSTAL_VALUES
]

In [6]:
unblind_script_path = "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks/unblind/"
import sys
sys.path.append(unblind_script_path)
from histogram_maker import *
from do_scan_analysis import *

In [ ]:
import numpy as np
from NNMFit.utilities import load_pickle

# Per-component histograms are calculated WITHOUT Snowstorm gradients: the gradients add a fixed
# per-bin shift (also when all norms are zero) derived from neutrino MC. At plot time this shift
# (total_syst - total_nosyst) is distributed over the neutrino components only, the muon template
# (and its conv/prompt parts) is never affected.

for cfg in CONFIGS:
    name = cfg["name"]
    output_dir = os.path.join(main_out_dir, name)
    os.system(f"mkdir -p {output_dir}")
    outfile = os.path.join(output_dir,"Precalculated_Graph.pickle")

    print(10*"-")
    print(name)
    print(output_dir)
    print(outfile)

    NNMFit_config_options = {
        "init_method": "from_configs",
        "main_config": os.path.join(configs_dir, cfg["main_config"]),
        "analysis_config": os.path.join(configs_dir, cfg["analysis_config"]),
        "config_dir": configs_dir,
        "override_configs": cfg["override_configs"] + cfg["systematics"],
        "override_components": cfg["override_components"],
        "override_parameters": cfg["override_parameters"],
    }

    cmd = [
        python, script,
        "--main_config", os.path.join(configs_dir, cfg["main_config"]),
        "--analysis_config", os.path.join(configs_dir, cfg["analysis_config"]),
        "--config_dir", configs_dir,
        "--override_configs", *cfg["override_configs"] + cfg["systematics"],
        "--override_components", *cfg["override_components"],
        "-o", outfile,
    ]
    if cfg["override_parameters"]:
        cmd += ["--override_parameters", *cfg["override_parameters"]]

    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)

    # Data histogram from the precalculated graph, with make_histogram.py of the same venv
    data_hist_file = os.path.join(output_dir, "Data_Histogram.pickle")
    result = subprocess.run(
        [python, make_histogram_script, "--precalculated_graph", outfile, "--outfile", data_hist_file],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr)
    data_hist = generate_data_hist(scan_path=output_dir)

    # the data histogram must agree with the one of the fit (if the same data sample is used)
    if cfg["fit_dir"] is not None:
        fit_data_hist = load_pickle(os.path.join(cfg["fit_dir"], "Data_Histogram.pickle"))
    for det_config, h in data_hist["histograms"].items():
        same = np.array_equal(h, fit_data_hist["histograms"][det_config]) if cfg["fit_dir"] is not None else "not compared"
        print(f"Data histogram {det_config}: {h.sum():.0f} events, identical to fit: {same}")

    # Total histogram WITH Snowstorm gradients (systematics)
    make_histogram_from_variables(NNMFit_config_options=NNMFit_config_options, out_file=f"{output_dir}/MC_Histogram.pickle")

    # Total histogram WITHOUT Snowstorm gradients — used to compute per-bin gradient at plot time
    NNMFit_config_options_nosyst = {
        **NNMFit_config_options,
        "override_configs": cfg["override_configs"] + DEFAULT_NO_SYSTEMATICS,
    }
    make_histogram_from_variables(NNMFit_config_options=NNMFit_config_options_nosyst, out_file=f"{output_dir}/MC_Histogram_nosyst.pickle")

    # Per-component histograms WITHOUT Snowstorm gradients, one analysis config per component
    # (written by run_post_fit_scripts.ipynb, all other norms set to zero)
    for component, config_suffix in cfg["components"].items():
        analysis_config_comp = os.path.join(configs_dir, cfg["analysis_config"].replace(".yaml", f"_{config_suffix}.yaml"))
        if not os.path.exists(analysis_config_comp):
            print(f"Skipping {component}: config not found at {analysis_config_comp}")
            continue
        opts = {
            **NNMFit_config_options_nosyst,
            "analysis_config": analysis_config_comp,
        }
        print(10*"-", "Component (no syst)", component)
        print(analysis_config_comp)
        make_histogram_from_variables(NNMFit_config_options=opts, out_file=f"{output_dir}/MC_Histogram_{component}_nosyst.pickle")

In [ ]:
# Consistency checks (no systematics): the components add up to the total, the muon conv/prompt parts
# add up to the muon template, and for the 3-flavor graph the flavors add up to the astro component and
# the totals agree with the graph of the fit parametrization
# largest per-bin difference relative to the channel total (per-bin relative differences are
# meaningless in bins where the Snowstorm shift cancels the prediction to ~0)
rel = lambda x, y: np.max(np.abs(x - y)) / max(np.sum(y), 1e-300)
ref_dir = os.path.join(main_out_dir, CONFIGS[0]["name"])

for cfg in CONFIGS:
    output_dir = os.path.join(main_out_dir, cfg["name"])
    hist = lambda d, n: load_pickle(os.path.join(d, f"MC_Histogram{n}.pickle"))["histograms"]
    total_syst, total_nosyst = hist(output_dir, ""), hist(output_dir, "_nosyst")
    comps = {c: hist(output_dir, f"_{c}_nosyst") for c in cfg["components"]}

    print(cfg["name"])
    for det_config in total_nosyst:
        sum_comps = sum(comps[c][det_config] for c in ["Astro", "Conv", "Prompt", "Muon"])
        sum_muon = comps["MuonConv"][det_config] + comps["MuonPrompt"][det_config]
        print(f"  {det_config}")
        print("    " + ", ".join(f"{c} {comps[c][det_config].sum():.2f}" for c in comps))
        print(f"    total nosyst {total_nosyst[det_config].sum():.2f}, total syst {total_syst[det_config].sum():.2f} "
              f"(Snowstorm shift {np.sum(total_syst[det_config] - total_nosyst[det_config]):+.2f})")
        print(f"    max diff/total  Astro+Conv+Prompt+Muon vs total: {rel(sum_comps, total_nosyst[det_config]):.1e}, "
              f"MuonConv+MuonPrompt vs Muon: {rel(sum_muon, comps['Muon'][det_config]):.1e}")
        if "_ice_crystal_" in cfg["name"]:
            # ice_crystal only enters the Snowstorm shift
            print(f"    max diff/total  vs best fit graph, total syst / nosyst: "
                  f"{rel(total_syst[det_config], hist(ref_dir, '')[det_config]):.1e} / "
                  f"{rel(total_nosyst[det_config], hist(ref_dir, '_nosyst')[det_config]):.1e}")
        if "Astro_NuE" in comps:
            sum_flavor = sum(comps[c][det_config] for c in ["Astro_NuE", "Astro_NuMu", "Astro_NuTau"])
            print(f"    max diff/total  NuE+NuMu+NuTau vs Astro: {rel(sum_flavor, comps['Astro'][det_config]):.1e}, "
                  f"total syst / nosyst vs fit parametrization: "
                  f"{rel(total_syst[det_config], hist(ref_dir, '')[det_config]):.1e} / "
                  f"{rel(total_nosyst[det_config], hist(ref_dir, '_nosyst')[det_config]):.1e}")